# 00_setup - ElectroCasa

Ejecuta este notebook **antes del despliegue del bundle**.

Crea:

- catálogos `electrocasa_dev` y `electrocasa_prod`
- schemas `landing`, `bronze`, `silver`, `gold` y `audit`
- Volume `bronze.landing`
- permisos diferenciados para Ingeniería, Analistas y Auditoría

## Importante para Databricks Free Edition

Los grupos que se usarán con Unity Catalog deben ser **account groups**.

No uses `CREATE GROUP` por SQL: ese comando crea grupos locales del workspace y
esos grupos no son válidos para permisos de Unity Catalog.

Free Edition no expone account-level APIs ni SCIM. Por eso, antes de ejecutar este
notebook crea manualmente estos tres grupos desde:

**Settings > Identity and access > Groups > Manage > Add Group > Add new**

- `electrocasa_ingenieria`
- `electrocasa_analistas`
- `electrocasa_auditoria`

Luego agrega tu usuario a `electrocasa_ingenieria`.


In [ ]:
CATALOGOS = ["electrocasa_dev", "electrocasa_prod"]

GRUPO_ING = "electrocasa_ingenieria"
GRUPO_ANALISTAS = "electrocasa_analistas"
GRUPO_AUDITORIA = "electrocasa_auditoria"

print("Grupos esperados:")
print("-", GRUPO_ING)
print("-", GRUPO_ANALISTAS)
print("-", GRUPO_AUDITORIA)


In [ ]:
# Catálogos, schemas y Volume.
for catalogo in CATALOGOS:
    spark.sql(f"CREATE CATALOG IF NOT EXISTS {catalogo}")

    for schema in ["landing", "bronze", "silver", "gold", "audit"]:
        spark.sql(
            f"CREATE SCHEMA IF NOT EXISTS {catalogo}.{schema}"
        )

    spark.sql(
        f"CREATE VOLUME IF NOT EXISTS {catalogo}.bronze.landing"
    )

    print(f"Aprovisionado: {catalogo}")


In [ ]:
# Permisos diferenciados.
# Si alguno de los account groups no existe, esta celda fallará de forma clara.
for catalogo in CATALOGOS:
    # Ingeniería: lectura/escritura de las capas del proyecto.
    spark.sql(
        f"GRANT USE CATALOG ON CATALOG {catalogo} TO `{GRUPO_ING}`"
    )

    for schema in ["landing", "bronze", "silver", "gold", "audit"]:
        spark.sql(
            f"GRANT USE SCHEMA ON SCHEMA {catalogo}.{schema} "
            f"TO `{GRUPO_ING}`"
        )
        spark.sql(
            f"GRANT SELECT, MODIFY, CREATE TABLE "
            f"ON SCHEMA {catalogo}.{schema} TO `{GRUPO_ING}`"
        )

    spark.sql(
        f"GRANT READ VOLUME, WRITE VOLUME "
        f"ON VOLUME {catalogo}.bronze.landing TO `{GRUPO_ING}`"
    )

    # Analistas: solo lectura de Gold.
    spark.sql(
        f"GRANT USE CATALOG ON CATALOG {catalogo} "
        f"TO `{GRUPO_ANALISTAS}`"
    )
    spark.sql(
        f"GRANT USE SCHEMA ON SCHEMA {catalogo}.gold "
        f"TO `{GRUPO_ANALISTAS}`"
    )
    spark.sql(
        f"GRANT SELECT ON SCHEMA {catalogo}.gold "
        f"TO `{GRUPO_ANALISTAS}`"
    )

    # Auditoría: lectura de Gold + objetos de auditoría y metadatos navegables.
    spark.sql(
        f"GRANT USE CATALOG, BROWSE ON CATALOG {catalogo} "
        f"TO `{GRUPO_AUDITORIA}`"
    )

    for schema in ["gold", "audit"]:
        spark.sql(
            f"GRANT USE SCHEMA ON SCHEMA {catalogo}.{schema} "
            f"TO `{GRUPO_AUDITORIA}`"
        )
        spark.sql(
            f"GRANT SELECT ON SCHEMA {catalogo}.{schema} "
            f"TO `{GRUPO_AUDITORIA}`"
        )

    print(f"Permisos aplicados: {catalogo}")


In [ ]:
# Comprobación simple de que el Volume existe.
for catalogo in CATALOGOS:
    ruta = f"/Volumes/{catalogo}/bronze/landing"
    print(ruta)
    display(dbutils.fs.ls(ruta))


## Siguiente paso

Sube los cinco archivos entregados a las carpetas del Volume del target que vas a probar.

Para `dev`:

- `ventas_sucursales.csv` → `/Volumes/electrocasa_dev/bronze/landing/ventas/`
- `catalogo_productos.json` → `/Volumes/electrocasa_dev/bronze/landing/catalogo/`
- `empleados_rrhh.csv` → `/Volumes/electrocasa_dev/bronze/landing/empleados/`
- `resenas_clientes.json` → `/Volumes/electrocasa_dev/bronze/landing/resenas/`
- `devoluciones.csv` → `/Volumes/electrocasa_dev/bronze/landing/devoluciones/`

`tracking_envios_azure_sql.sql` **no se sube al Volume**. El tracking se consulta desde
la Azure SQL Database mediante Lakehouse Federation.
